In [13]:
IMAGE_OUTPUT = "data/dataset/images/sam3_images"
EXAMPLE_IMAGE = "data/dataset/dummy_images/imagenReal.jpg"
MODEL_PATH = "data/dataset/sam3_model/sam3.pt"
CLASIFICATOR_DATA = "../data/dataset/clasificator_data"

In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
# hay que importar python -m pip install -U pip huggingface_hub y logearse con tu token
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

image_path = "../data/dataset/dummy_images/imagenReal2.jpeg"
image = Image.open(image_path).convert("RGB")

inputs = processor(images=image, text="leaves", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")


In [19]:
def procesar_directorio_completo(directorio_raiz_imagenes, directorio_salida_etiquetas, model, processor, device, clasificator_data=False):
    for root, dirs, files in os.walk(directorio_raiz_imagenes):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                ruta_imagen = os.path.join(root, file)

                print(f"Analizando: {ruta_imagen}")
                etiquetar_imagen_con_sam(str(ruta_imagen), directorio_salida_etiquetas, model, processor, device, clasificator_data)


In [ ]:
import torch
from transformers import Sam3Processor, Sam3Model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"iniciando proceso en: {device}")

print("cargando el modelo SAM 3...")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("modelo cargado correctamente.")

DIRECTORIO_IMAGENES = "../data/dataset/images/sam3"
DIRECTORIO_ETIQUETAS = "../data/dataset/labels/sam3"

procesar_directorio_completo(DIRECTORIO_IMAGENES, DIRECTORIO_ETIQUETAS, model, processor, device, True)

print("LO HEMOS CONSEGUIDO, PROCESO DE ETIQUETADO COMPLETADO.")

In [20]:
import torch

def etiquetar_imagen_con_sam(
    ruta_imagen,
    directorio_etiquetas,
    model,
    processor,
    device,
    clasificator_data
):
    image = Image.open(ruta_imagen).convert("RGB")

    image.thumbnail((1024, 1024))
    img_width, img_height = image.size

    inputs = processor(images=image, text="leaf", return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.6,
        mask_threshold=0.5,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]

    bboxes_sam = results['boxes'].tolist()
    scores = results['scores'].tolist()
    bboxes_yolo = []

    for bbox in bboxes_sam:
        bbox_yolo = convertir_sam_a_yolo(bbox, img_width, img_height)
        bboxes_yolo.append(bbox_yolo)

    if bboxes_yolo:
        nombre_archivo = os.path.splitext(os.path.basename(ruta_imagen))[0]

        guardar_etiquetas_yolo(
            ruta_original_imagen=ruta_imagen,
            directorio_salida=directorio_etiquetas,
            nombre_archivo=nombre_archivo,
            bboxes_yolo=bboxes_yolo
        )

        if clasificator_data:

            nombre_minusculas = nombre_archivo.lower()

            print(f"nombre del archivo: {nombre_minusculas}")

            if "sano" in nombre_minusculas:
                subcarpeta = CLASIFICATOR_DATA + "/healthy"
                print("es sana")
            elif "enferma" in nombre_minusculas or "enfermo" in nombre_minusculas:
                subcarpeta = CLASIFICATOR_DATA + "/diseased"
                print("es enferma")
            else:
                print("es sin clasificar, se omite")
                return

            os.makedirs(subcarpeta, exist_ok=True)
            cajas_con_scores = list(zip(bboxes_sam, scores))
            cajas_con_scores.sort(key=lambda x: x[1], reverse=True)
            top_5 = cajas_con_scores[:5]

            for index, (bbox, score) in enumerate(top_5):
                xmin, ymin, xmax, ymax = bbox

                recorte = image.crop((xmin, ymin, xmax, ymax))

                nombre_recorte = f"{nombre_archivo}_top{index+1}_score{score:.2f}.jpg"
                ruta_guardado = os.path.join(subcarpeta, nombre_recorte)

                recorte.save(ruta_guardado)

    else:
        print(f"no se detectaron hojas en: {ruta_imagen}")

In [7]:
def convertir_sam_a_yolo(bbox_sam, img_width, img_height):
    x_min, y_min, x_max, y_max = bbox_sam

    centro_x = (x_min + x_max) / 2.0
    centro_y = (y_min + y_max) / 2.0

    ancho = x_max - x_min
    alto = y_max - y_min

    centro_x_norm = centro_x / img_width
    centro_y_norm = centro_y / img_height
    ancho_norm = ancho / img_width
    alto_norm = alto / img_height

    return [round(centro_x_norm, 6), round(centro_y_norm, 6), round(ancho_norm, 6), round(alto_norm, 6)]

In [8]:
def guardar_etiquetas_yolo(ruta_original_imagen, directorio_salida, nombre_archivo, bboxes_yolo, clase_id=0):

    ruta_carpeta_origen = os.path.dirname(ruta_original_imagen)
    nombre_carpeta_origen = os.path.basename(ruta_carpeta_origen)

    # Si el archivo empieza por "frame", añadimos esa subcarpeta al destino para que la estructura de images y labels sea la misma, lo que facilita el entrenamiento con YOLO
    if nombre_archivo.startswith('frame'):
        directorio_salida = os.path.join(directorio_salida, nombre_carpeta_origen)

    os.makedirs(directorio_salida, exist_ok=True)

    ruta_archivo = os.path.join(directorio_salida, f"{nombre_archivo}.txt")
    with open(ruta_archivo, 'w') as archivo:
        for bbox in bboxes_yolo:
            cx, cy, w, h = bbox
            linea = f"{clase_id} {cx} {cy} {w} {h}\n"
            archivo.write(linea)

    print(f"Etiquetas guardadas en: {ruta_archivo}")

In [9]:
import cv2
import numpy as np

def extraer_fotogramas_representativos(video_path, output_path, threshold=30.0):

    # Creamos el directorio de salida si no existe
    os.makedirs(output_path, exist_ok=True)
    nombre_archivo = os.path.splitext(os.path.basename(video_path))[0]

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"error al abrir el vídeo: {video_path}")
        return

    # Leemos el primer fotograma
    ret, prev_frame = cap.read()
    if not ret:
        print("el vídeo está vacío o no se puede leer.")
        return

    saved_count = 0

    # Guardamos el primer fotograma por defecto
    # le cambiamos el nombre del primero para si queremos extraer fotogramas para el entrenamiento del clasificador sepa si es enfermo o sano
    cv2.imwrite(os.path.join(output_path, f"{nombre_archivo}_frame_{saved_count:04d}.jpg"), prev_frame)
    saved_count += 1

    # Convertimos a escala de grises para la comparación
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculamos la diferencia absoluta entre el fotograma actual y el último que analicemos
        diff = cv2.absdiff(prev_gray, gray)
        mean_diff = np.mean(diff)

        # Si la diferencia supera el umbral, consideramos el fotograma representativo
        if mean_diff > threshold:
            cv2.imwrite(os.path.join(output_path, f"frame_{saved_count:04d}.jpg"), frame)
            prev_gray = gray
            saved_count += 1

    cap.release()
    print(f"proceso completado. Se han extraído {saved_count} imágenes representativas.")


In [10]:
from PIL import Image
import pillow_heif

pillow_heif.register_heif_opener()

def convertir_heic_a_jpg(ruta_heic, ruta_salida_jpg):

    try:
        imagen = Image.open(ruta_heic)
        imagen_rgb = imagen.convert('RGB')

        imagen_rgb.save(ruta_salida_jpg, format="JPEG")
        print(f"imagen convertida y guardada en: {ruta_salida_jpg}")

    except Exception as e:
        print(f"error al procesar la imagen {ruta_heic}: {e}")

In [ ]:
import os
destination = "pre-trainingsetSam3/processed"
source = "pre-trainingsetSam3/non-processed"
print(f"Archivos de salida: {os.listdir(destination)}")
print(f"Archivos de salida: {os.listdir(source)}")
for root, dirs, files in os.walk(source):
    for file in files:
        extension = file.lower()
        file_path = os.path.join(root, file)

        if extension.endswith('.mov'):
            print(f"procesando vídeo: {file}")

            # creamos subcarpetas para tener mas organizadas las fotos de los videos
            nombre_video = os.path.splitext(file)[0]
            ruta_salida_video = os.path.join(destination, nombre_video)

            # llamamos a la función que creamos en el paso anterior
            extraer_fotogramas_representativos(str(file_path), str(ruta_salida_video), threshold=50.0)

        elif extension.endswith('.heic'):
            print(f"archivo HEIC encontrado: {file_path}")
            nombre_foto = os.path.splitext(file)[0]
            ruta_salida_jpg = os.path.join(destination, nombre_foto+".jpg")
            convertir_heic_a_jpg(str(file_path), str(ruta_salida_jpg))